# 07.5 - Word Embeddings (Word2Vec Concepts)

**Phase:** 07 - NLP

**Status:** VERIFIED

---

## 1. What Are We Solving?

BoW/TF-IDF treat every word as an independent symbol ('cat' is unrelated to 'dog'). **Word embeddings** are dense, low-dimensional vectors where words used in similar contexts end up close together — capturing meaning.

## 2. Why Does This Matter?

The **distributional hypothesis** — 'you shall know a word by the company it keeps' — lets embeddings capture gender, tense, plural, and topic. They are the foundation of all modern NLP and power word analogies like king - man + woman ~ queen.

## 3. Prerequisites

- Units 07.1-07.4
- Phase 06 (neural networks, PyTorch basics)

## 4. Learning Objectives

By the end of this unit, you should be able to:
- Explain Skip-gram vs CBOW
- Implement a small CBOW/Skip-gram embedding model in PyTorch from scratch
- Compute cosine similarity between word vectors
- Evaluate embedding quality qualitatively

## 5. Mental Model

```text
Sparse one-hot:  cat = [0,0,0,1,0,0,...]  (no meaning)
Dense embedding: cat = [0.2, -0.5, 0.8, 0.1]  (carries meaning)
```
The hidden layer weights of a shallow network (predict context from word, or word from context) become the embeddings.


## 6. Setup

CPU-only PyTorch; small model, tiny corpus so it runs fast.


In [1]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from collections import Counter

torch.manual_seed(0)
print('torch', torch.__version__)


torch 2.13.0+cpu


## 7. Tiny Corpus & Vocabulary


In [2]:
sentences = [
    "the cat sat on the mat".split(),
    "the dog sat on the log".split(),
    "cats and dogs are friends".split(),
    "the cat chased the dog quickly".split(),
    "the dog chased the cat slowly".split(),
]

vocab = sorted({w for s in sentences for w in s})
w2i = {w: i for i, w in enumerate(vocab)}
i2w = {i: w for w, i in w2i.items()}
print("Vocabulary:", vocab)
print("Vocab size:", len(vocab))


Vocabulary: ['and', 'are', 'cat', 'cats', 'chased', 'dog', 'dogs', 'friends', 'log', 'mat', 'on', 'quickly', 'sat', 'slowly', 'the']
Vocab size: 15


## 8. Build (context, target) Pairs (CBOW)

CBOW: given context words, predict the target word in the middle.


In [3]:
window = 2
pairs = []
for s in sentences:
    idx = [w2i[w] for w in s]
    for pos, target in enumerate(idx):
        ctx = idx[max(0, pos-window):pos] + idx[pos+1:pos+window+1]
        # pad context to fixed length 2*window
        ctx = ctx + [w2i['the']] * (2*window - len(ctx))
        pairs.append((ctx, target))

print("Number of training pairs:", len(pairs))
for ctx, tgt in pairs[:6]:
    print([i2w[c] for c in ctx], "->", i2w[tgt])


Number of training pairs: 29
['cat', 'sat', 'the', 'the'] -> the
['the', 'sat', 'on', 'the'] -> cat
['the', 'cat', 'on', 'the'] -> sat
['cat', 'sat', 'the', 'mat'] -> on
['sat', 'on', 'mat', 'the'] -> the
['on', 'the', 'the', 'the'] -> mat


## 9. The CBOW Model

A tiny neural net that averages context embeddings and predicts a word.


In [4]:
class CBOW(nn.Module):
    def __init__(self, vocab_size, embed_dim):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, embed_dim)
        self.fc = nn.Linear(embed_dim, vocab_size)

    def forward(self, context):
        # context: (batch, window*2)
        emb = self.embed(context)            # (batch, ctx_len, embed_dim)
        avg = emb.mean(dim=1)                # (batch, embed_dim)
        return self.fc(avg)                  # (batch, vocab_size)

model = CBOW(len(vocab), embed_dim=30)
print(model)


CBOW(
  (embed): Embedding(15, 30)
  (fc): Linear(in_features=30, out_features=15, bias=True)
)


## 10. Train the CBOW Model

CPU-friendly: small data, few epochs.


In [5]:
contexts = torch.tensor([p[0] for p in pairs])
targets = torch.tensor([p[1] for p in pairs])

optimizer = torch.optim.Adam(model.parameters(), lr=0.05)
loss_fn = nn.CrossEntropyLoss()

for epoch in range(200):
    model.train()
    optimizer.zero_grad()
    logits = model(contexts)
    loss = loss_fn(logits, targets)
    loss.backward()
    optimizer.step()
    if epoch % 40 == 0:
        print(f"epoch {epoch:3d}: loss={loss.item():.3f}")
print("Done. Final loss:", round(loss.item(), 3))


epoch   0: loss=2.748


epoch  40: loss=0.154


epoch  80: loss=0.146
epoch 120: loss=0.145


epoch 160: loss=0.145


Done. Final loss: 0.144


## 11. Extract Embeddings & Cosine Similarity

The `embed` layer weights are the word embeddings.


In [6]:
embeddings = model.embed.weight.detach().numpy()  # (vocab, dim)

def cos_sim(a, b):
    a = a / (np.linalg.norm(a) + 1e-8)
    b = b / (np.linalg.norm(b) + 1e-8)
    return float(a @ b)

w = lambda s: embeddings[w2i[s]]

pairs_to_check = [('cat', 'dog'), ('the', 'and'), ('cat', 'mat'), ('dog', 'log')]
for a, b in pairs_to_check:
    print(f"cos({a}, {b}) = {cos_sim(w(a), w(b)):.3f}")

print("\nWords sharing contexts (cat/dog, mat/log) tend to have higher similarity.")


cos(cat, dog) = 0.620
cos(the, and) = -0.243
cos(cat, mat) = 0.267
cos(dog, log) = 0.556

Words sharing contexts (cat/dog, mat/log) tend to have higher similarity.


## 12. Analogies: king - man + woman ~ queen

We test a (very rough, tiny-corpus) analogy using vector arithmetic.


In [7]:
def most_similar(vec, topn=3):
    sims = [(cos_sim(vec, embeddings[i]), i2w[i]) for i in range(len(vocab))]
    sims.sort(reverse=True)
    return [(w, round(s, 3)) for s, w in sims[:topn]]

# naive analogy: 'cat' - 'the' + 'dog'
vec = w('cat') - w('the') + w('dog')
print("cat - the + dog ->", most_similar(vec))

print("\nWith a tiny corpus analogies are noisy; with millions of words they become clean.")


cat - the + dog -> [('cat', 0.906), ('dog', 0.823), ('log', 0.56)]

With a tiny corpus analogies are noisy; with millions of words they become clean.


## 13. Visualize Embeddings (2D PCA)


In [8]:
from sklearn.decomposition import PCA

pca = PCA(n_components=2)
proj = pca.fit_transform(embeddings)

fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(proj[:, 0], proj[:, 1])
for i, word in enumerate(vocab):
    ax.annotate(word, (proj[i, 0], proj[i, 1]), fontsize=9)
ax.set_title("CBOW word embeddings projected to 2D (PCA)")
plt.tight_layout()
plt.savefig('07_05_embeddings.png', dpi=90)
print("Saved scatter plot. Similar words should cluster loosely.")


Saved scatter plot. Similar words should cluster loosely.


## 14. Failure Case: Tiny Corpus Produces Noise

Embeddings need huge corpora (millions of words). Our toy corpus yields noisy vectors.


In [9]:
print("Real Word2Vec was trained on ~1.6B tokens (Google News).")
print("Our corpus has", sum(len(s) for s in sentences), "tokens.")
print("\nLesson: for small data, prefer pretrained embeddings (GloVe/Word2Vec) or skip embeddings.")
print("For OOV/morphology, use FastText subword embeddings.")


Real Word2Vec was trained on ~1.6B tokens (Google News).
Our corpus has 29 tokens.

Lesson: for small data, prefer pretrained embeddings (GloVe/Word2Vec) or skip embeddings.
For OOV/morphology, use FastText subword embeddings.


## 15. Debugging: Common Errors

- **Random-looking similarities** — data too small. Fix: bigger corpus or pretrained.
- **All similarities ~1.0** — vectors degenerate/un-normalized. Fix: normalize, retrain.
- **OOV words return zero** — word not in vocab. Fix: subword (FastText) or average known tokens.
- **No meaning captured** — window too small/large. Fix: window=5 general, window=2 syntax.

## 16. Real-World Considerations

- Use pretrained embeddings unless you have millions of words.
- `vector_size` 100-300; window=5 starting point.
- Evaluate embeddings qualitatively (similar words) and downstream (task accuracy).

## 17. Common Mistakes

- Small corpus, expecting good embeddings.
- `vector_size` too high for small data.
- Confusing similarity with 'relatedness'.

## 18. When NOT to Use

- Very small domain corpora (use pretrained or TF-IDF).
- When you need interpretable features.

## 19. Challenge

Implement skip-gram (predict context from a center word) and compare with CBOW.


In [10]:
# Challenge: skip-gram pairs (center -> contexts)
sg_pairs = []
for s in sentences:
    idx = [w2i[w] for w in s]
    for pos, center in enumerate(idx):
        ctx = idx[max(0,pos-window):pos] + idx[pos+1:pos+window+1]
        for c in ctx:
            sg_pairs.append((center, c))

class SkipGram(nn.Module):
    def __init__(self, vocab_size, embed_dim):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, embed_dim)
        self.out = nn.Linear(embed_dim, vocab_size)
    def forward(self, center):
        return self.out(self.embed(center))

sg_model = SkipGram(len(vocab), 30)
centers = torch.tensor([p[0] for p in sg_pairs])
ctxt    = torch.tensor([p[1] for p in sg_pairs])
opt = torch.optim.Adam(sg_model.parameters(), lr=0.02)
for epoch in range(100):
    opt.zero_grad()
    loss = F.cross_entropy(sg_model(centers), ctxt)
    loss.backward()
    opt.step()
print("SkipGram trained. #pairs:", len(sg_pairs), "final loss:", round(loss.item(), 3))
print("Skip-gram is better for rare words; CBOW is faster (average context).")


SkipGram trained. #pairs: 86 final loss: 1.425
Skip-gram is better for rare words; CBOW is faster (average context).


## 20. Closed-Book Recall

1. What is the distributional hypothesis?
2. Difference between Skip-gram and CBOW?
3. What metric measures embedding similarity?
4. Why not train embeddings on a 100-word corpus?

## 21. Teach-Back Questions

- Explain why 'cat' and 'dog' end up close.
- Explain the king - man + woman analogy in principle.

## 22. Summary

You implemented a from-scratch CBOW (and skip-gram) embedding in PyTorch on a tiny corpus, computed cosine similarity, and visualized embeddings. This is the concept behind all modern NLP.

## 23. Further Experiment

- Use embeddings as input features to the classifier from 07.4 and compare vs TF-IDF.
- Train on a larger corpus (e.g., book text) for cleaner vectors.

## 24. Verification Status

```
STATUS: VERIFIED
EXECUTION: PASS
DEPENDENCIES: numpy, torch, scikit-learn, matplotlib
OUTPUTS: PASS
LAST VERIFIED: 2026-08-29
```
